# Fish Audio S2 Pro Colab Experiment

Self-contained Colab workflow for Fish Audio S2 Pro TTS / authorized voice cloning.

- This notebook clones `https://github.com/fishaudio/fish-speech` directly and runs Fish Speech from that checkout.
- It does not clone this `qwen-asr-eval` repo and does not depend on the local experiment helper package.
- HF model files stay in ephemeral `/content`; Google Drive stores only generated outputs, logs, manifests, and optional reference copies.
- Use only voices/audio you are authorized to clone or synthesize.


In [ ]:
#@title 0. Runtime configuration and inline helpers
from pathlib import Path
import json, os, re, shlex, shutil, signal, subprocess, sys, time, urllib.request

FISH_SPEECH_REPO_URL = "https://github.com/fishaudio/fish-speech.git"  #@param {type:"string"}
FISH_SPEECH_REF = "02995ed7bd61ea383727a2c173a3ce126965219c"  #@param {type:"string"}
FISH_REPO = Path("/content/fish-speech")
WORK_ROOT = Path("/content/fishaudio_s2_pro")
DRIVE_ROOT = Path("/content/drive/MyDrive/voice/fishaudio-s2-pro")
RUN_ID = ""  #@param {type:"string"}
MOUNT_DRIVE = True  #@param {type:"boolean"}

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"This experiment targets Python 3.12; current kernel is {sys.version}")

def run(cmd, *, cwd=None, env=None, check=True):
    started = time.time()
    display_cwd = str(cwd) if cwd else os.getcwd()
    print(f"\n[{time.strftime('%H:%M:%S')}] cwd={display_cwd}", flush=True)
    print("$", " ".join(map(str, cmd)), flush=True)
    proc_env = os.environ.copy()
    if env:
        proc_env.update(env)
    proc_env["PYTHONUNBUFFERED"] = "1"
    proc_env.setdefault("PIP_PROGRESS_BAR", "on")
    proc_env.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)
    result = subprocess.run(cmd, cwd=cwd, env=proc_env, check=check)
    print(f"[{time.strftime('%H:%M:%S')}] done in {time.time() - started:.1f}s", flush=True)
    return result

def default_run_id(prefix="fishaudio_s2_pro"):
    return f"{prefix}_{time.strftime('%Y%m%d_%H%M%S')}"

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return path

def read_tail(path, lines=60):
    path = Path(path)
    if not path.exists():
        return ""
    return "\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])

def read_colab_secret(name, *, required=True):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Could not read Colab Secret {name}: {exc!r}") from exc
        return None
    if required and not value:
        raise RuntimeError(f"Missing Colab Secret: {name}")
    return value

def gpu_inventory():
    result = subprocess.run([
        "nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits",
    ], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
    gpus = []
    for line in result.stdout.splitlines():
        if not line.strip() or "," not in line:
            continue
        name, memory_mb = [part.strip() for part in line.split(",", 1)]
        gpus.append({"name": name, "memory_mb": int(memory_mb)})
    return gpus

def require_minimum_vram(minimum_gb=24):
    gpus = gpu_inventory()
    if not gpus:
        raise RuntimeError("No NVIDIA GPU detected. Use a Colab A100/H100 runtime.")
    best_mb = max(int(gpu["memory_mb"]) for gpu in gpus)
    if best_mb < minimum_gb * 1024:
        raise RuntimeError(f"Fish Audio S2 Pro needs at least {minimum_gb} GB VRAM; largest detected GPU has {best_mb / 1024:.1f} GB.")
    return gpus

def resolve_fish_uv_extra(value="auto"):
    if value != "auto":
        if value not in {"cu126", "cu128", "cu129", "cpu"}:
            raise ValueError("FISH_UV_EXTRA must be auto, cu126, cu128, cu129, or cpu")
        return value
    result = subprocess.run(["nvidia-smi"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
    match = re.search(r"CUDA Version:\s*([0-9]+)\.([0-9]+)", result.stdout or "")
    if not match:
        return "cu126"
    version = (int(match.group(1)), int(match.group(2)))
    if version >= (12, 9):
        return "cu129"
    if version >= (12, 8):
        return "cu128"
    return "cu126"

def wait_for_http(url, *, timeout_seconds=600, interval_seconds=2.0):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=10) as response:
                if 200 <= response.status < 500:
                    return True
        except Exception as exc:
            last_error = exc
            time.sleep(interval_seconds)
    raise RuntimeError(f"Timed out waiting for {url}: {last_error!r}")

def start_process(name, command, *, cwd, log_path, env=None, display_command=None):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    proc_env = os.environ.copy()
    if env:
        proc_env.update(env)
    proc_env.setdefault("PYTHONUNBUFFERED", "1")
    with log_path.open("ab") as log_file:
        proc = subprocess.Popen(
            command,
            cwd=Path(cwd),
            env=proc_env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    info = {"name": name, "pid": proc.pid, "command": display_command or command, "log_path": str(log_path)}
    print(f"Started {name}: pid={proc.pid} log={log_path}", flush=True)
    return info

def stop_process(proc_info, *, timeout_seconds=30):
    if not proc_info:
        return
    pid = int(proc_info["pid"])
    try:
        os.killpg(pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            os.killpg(pid, 0)
        except ProcessLookupError:
            return
        time.sleep(0.5)
    try:
        os.killpg(pid, signal.SIGKILL)
    except ProcessLookupError:
        return

def fish_env():
    env = os.environ.copy()
    env.update({
        "HF_HOME": str(CACHE_ROOT / "hf"),
        "HF_HUB_CACHE": str(CACHE_ROOT / "hf" / "hub"),
        "TRANSFORMERS_CACHE": str(CACHE_ROOT / "hf" / "transformers"),
        "PYTHONUNBUFFERED": "1",
    })
    return env

def start_api_server(*, port=8080, compile_model=False, half=False, api_key=None):
    command = [
        "uv", "run", "python", "tools/api_server.py",
        "--listen", f"127.0.0.1:{port}",
        "--llama-checkpoint-path", str(CHECKPOINT_DIR),
        "--decoder-checkpoint-path", str(CHECKPOINT_DIR / "codec.pth"),
        "--decoder-config-name", "modded_dac_vq",
        "--device", "cuda",
        "--workers", "1",
    ]
    if compile_model:
        command.append("--compile")
    if half:
        command.append("--half")
    if api_key:
        command.extend(["--api-key", api_key])
    display = ["REDACTED" if part == api_key else part for part in command]
    return start_process("fish-api", command, cwd=FISH_REPO, log_path=LOGS_DIR / f"fish_api_{port}.log", env=fish_env(), display_command=display)

def start_gradio_webui(*, compile_model=False, half=False, theme="light"):
    command = [
        "uv", "run", "python", "tools/run_webui.py",
        "--llama-checkpoint-path", str(CHECKPOINT_DIR),
        "--decoder-checkpoint-path", str(CHECKPOINT_DIR / "codec.pth"),
        "--decoder-config-name", "modded_dac_vq",
        "--device", "cuda",
        "--theme", theme,
    ]
    if compile_model:
        command.append("--compile")
    if half:
        command.append("--half")
    return start_process("fish-gradio", command, cwd=FISH_REPO, log_path=LOGS_DIR / "fish_gradio.log", env=fish_env())

def sanitize_voice_id(value):
    slug = re.sub(r"[^A-Za-z0-9_-]+", "-", value.strip()).strip("-")
    return slug[:80] or "voice"

print("Python:", sys.version)
print("Fish Speech target:", FISH_SPEECH_REPO_URL, FISH_SPEECH_REF)


In [ ]:
#@title 1. Mount Drive and clone Fish Speech directly
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if FISH_REPO.exists():
    run(["git", "remote", "set-url", "origin", FISH_SPEECH_REPO_URL], cwd=FISH_REPO, check=False)
    run(["git", "fetch", "origin", "--tags"], cwd=FISH_REPO)
else:
    run(["git", "clone", FISH_SPEECH_REPO_URL, str(FISH_REPO)])

if FISH_SPEECH_REF.strip():
    run(["git", "checkout", FISH_SPEECH_REF.strip()], cwd=FISH_REPO)

fish_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=FISH_REPO, text=True).strip()
required_paths = [
    FISH_REPO / "pyproject.toml",
    FISH_REPO / "tools" / "run_webui.py",
    FISH_REPO / "tools" / "api_server.py",
    FISH_REPO / "tools" / "server" / "views.py",
    FISH_REPO / "awesome_webui" / "package.json",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise RuntimeError("Fish Speech checkout is missing expected server/WebUI files: " + json.dumps(missing, indent=2))

print("Fish Speech repo:", FISH_REPO)
print("Fish Speech commit:", fish_commit)
print("Gradio WebUI:", FISH_REPO / "tools" / "run_webui.py")
print("Awesome WebUI source:", FISH_REPO / "awesome_webui")
print("Awesome WebUI backend route: /ui via tools/server/views.py")


In [ ]:
#@title 2. Configure run paths, GPU, secrets, and install Fish Speech dependencies
RUN_ID = RUN_ID.strip() or default_run_id()
RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID
LOGS_DIR = RUN_DIR / "logs"
OUTPUTS_DIR = RUN_DIR / "outputs"
MANIFESTS_DIR = RUN_DIR / "manifests"
CHECKPOINT_DIR = WORK_ROOT / "checkpoints" / "s2-pro"
CACHE_ROOT = WORK_ROOT / "cache"
for path in [RUN_DIR, LOGS_DIR, OUTPUTS_DIR, MANIFESTS_DIR, CHECKPOINT_DIR, CACHE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "HF_HOME": str(CACHE_ROOT / "hf"),
    "HF_HUB_CACHE": str(CACHE_ROOT / "hf" / "hub"),
    "TRANSFORMERS_CACHE": str(CACHE_ROOT / "hf" / "transformers"),
})

HF_TOKEN = read_colab_secret("HF_TOKEN", required=True)
os.environ["HF_TOKEN"] = HF_TOKEN

gpus = require_minimum_vram(24)
FISH_UV_EXTRA_REQUEST = "auto"  #@param ["auto", "cu126", "cu128", "cu129"]
FISH_UV_EXTRA = resolve_fish_uv_extra(FISH_UV_EXTRA_REQUEST)

write_json(MANIFESTS_DIR / "runtime.json", {
    "run_id": RUN_ID,
    "fish_repo": str(FISH_REPO),
    "fish_commit": fish_commit,
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "cache_root": str(CACHE_ROOT),
    "run_dir": str(RUN_DIR),
    "gpus": gpus,
    "fish_uv_extra": FISH_UV_EXTRA,
})

print("Run dir:", RUN_DIR)
print("Ephemeral checkpoint dir:", CHECKPOINT_DIR)
print("GPU inventory:", gpus)
print("Fish uv extra:", FISH_UV_EXTRA)

run(["apt-get", "-qq", "update"])
run(["apt-get", "-y", "-qq", "install", "ffmpeg", "portaudio19-dev", "libsox-dev"])
run([sys.executable, "-m", "pip", "install", "-U", "uv", "huggingface_hub", "requests"])

print("Installing Fish Speech upstream dependencies with uv. A fresh Colab VM can take several minutes.", flush=True)
run(["uv", "--color", "always", "sync", "--python", "3.12", "--extra", FISH_UV_EXTRA], cwd=FISH_REPO)


In [ ]:
#@title 3. Download Fish Audio S2 Pro weights to ephemeral /content
MODEL_ID = "fishaudio/s2-pro"
HF_DOWNLOAD_WORKERS = 2  #@param {type:"integer"}
download_log_path = LOGS_DIR / "hf_download.log"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not (CHECKPOINT_DIR / "codec.pth").exists():
    if shutil.which("hf") is None:
        raise RuntimeError("`hf` CLI was not found after installing huggingface_hub")

    print("Downloading S2 Pro weights to ephemeral /content.", flush=True)
    print("Download log:", download_log_path, flush=True)
    download_env = fish_env()
    download_env["HF_TOKEN"] = HF_TOKEN
    download_env["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    download_env.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)
    hf_cmd = [
        "hf", "download", MODEL_ID,
        "--local-dir", str(CHECKPOINT_DIR),
        "--max-workers", str(max(1, int(HF_DOWNLOAD_WORKERS))),
    ]
    shell_cmd = " ".join(shlex.quote(str(part)) for part in hf_cmd)
    shell_cmd = f"set -o pipefail; {shell_cmd} 2>&1 | tee -a {shlex.quote(str(download_log_path))}"
    run(["bash", "-lc", shell_cmd], cwd=FISH_REPO, env=download_env)
else:
    print("Checkpoint already present in ephemeral /content:", CHECKPOINT_DIR)

if not (CHECKPOINT_DIR / "codec.pth").exists():
    raise RuntimeError(f"Download completed but codec.pth is missing under {CHECKPOINT_DIR}")

write_json(MANIFESTS_DIR / "model_download.json", {"model_id": MODEL_ID, "checkpoint_dir": str(CHECKPOINT_DIR), "storage": "ephemeral_content"})


In [ ]:
#@title 4. Start upstream API server and run health check
API_PORT = 8080
COMPILE_MODEL = False  #@param {type:"boolean"}
USE_HALF = False  #@param {type:"boolean"}

api_proc = start_api_server(port=API_PORT, compile_model=COMPILE_MODEL, half=USE_HALF)
wait_for_http(f"http://127.0.0.1:{API_PORT}/v1/health", timeout_seconds=900)
print("API healthy:", f"http://127.0.0.1:{API_PORT}/v1/health")
print("API log tail:")
print(read_tail(api_proc["log_path"], lines=40))


In [ ]:
#@title 5. No-reference TTS smoke
from IPython.display import Audio, display
import requests

NO_REF_TEXT = "Hello from Fish Audio S2 Pro running on a Colab GPU."  #@param {type:"string"}
no_ref_output = OUTPUTS_DIR / "no_reference_smoke.wav"

payload = {
    "text": NO_REF_TEXT,
    "references": [],
    "reference_id": None,
    "format": "wav",
    "latency": "normal",
    "max_new_tokens": 1024,
    "chunk_length": 300,
    "top_p": 0.8,
    "repetition_penalty": 1.1,
    "temperature": 0.8,
    "streaming": False,
    "use_memory_cache": "off",
    "seed": 42,
}
started = time.time()
response = requests.post(f"http://127.0.0.1:{API_PORT}/v1/tts", json=payload, timeout=600)
elapsed = time.time() - started
if response.status_code != 200:
    error_path = no_ref_output.with_suffix(".error.json")
    write_json(error_path, {"status_code": response.status_code, "body": response.text[:4000], "payload": payload, "elapsed_seconds": elapsed})
    raise RuntimeError(f"TTS request failed with HTTP {response.status_code}; see {error_path}")
no_ref_output.write_bytes(response.content)
manifest = {"output_path": str(no_ref_output), "bytes": len(response.content), "elapsed_seconds": elapsed, "reference_id": None, "seed": 42, "text_chars": len(NO_REF_TEXT)}
write_json(no_ref_output.with_suffix(".wav.manifest.json"), manifest)
print(manifest)
display(Audio(filename=str(no_ref_output)))


In [ ]:
#@title 6. Prepare optional reference voice
REFERENCE_AUDIO_PATH = ""  #@param {type:"string"}
REFERENCE_TEXT = ""  #@param {type:"string"}
VOICE_ID = "demo_voice"  #@param {type:"string"}
REFERENCE_START_SECONDS = 0.0  #@param {type:"number"}
REFERENCE_MAX_SECONDS = 10.0  #@param {type:"number"}
SAVE_REFERENCE_COPY_TO_DRIVE = False  #@param {type:"boolean"}

voice_id = sanitize_voice_id(VOICE_ID)
reference_manifest = None
if REFERENCE_AUDIO_PATH.strip():
    source_audio = Path(REFERENCE_AUDIO_PATH.strip())
    if not source_audio.exists():
        raise FileNotFoundError(f"Reference audio does not exist: {source_audio}")
    if not REFERENCE_TEXT.strip():
        raise ValueError("REFERENCE_TEXT is required for voice cloning.")

    ref_dir = FISH_REPO / "references" / voice_id
    ref_dir.mkdir(parents=True, exist_ok=True)
    wav_path = ref_dir / "sample.wav"
    lab_path = ref_dir / "sample.lab"
    run([
        "ffmpeg", "-hide_banner", "-y",
        "-ss", str(REFERENCE_START_SECONDS),
        "-t", str(REFERENCE_MAX_SECONDS),
        "-i", str(source_audio),
        "-ac", "1", "-ar", "44100", "-vn", str(wav_path),
    ])
    lab_path.write_text(REFERENCE_TEXT.strip() + "\n", encoding="utf-8")

    copied_to_drive = False
    if SAVE_REFERENCE_COPY_TO_DRIVE:
        drive_ref = RUN_DIR / "references" / voice_id
        drive_ref.mkdir(parents=True, exist_ok=True)
        shutil.copy2(wav_path, drive_ref / wav_path.name)
        shutil.copy2(lab_path, drive_ref / lab_path.name)
        copied_to_drive = True

    reference_manifest = {
        "voice_id": voice_id,
        "source_audio": str(source_audio),
        "fish_reference_dir": str(ref_dir),
        "fish_reference_audio": str(wav_path),
        "fish_reference_text": str(lab_path),
        "start_seconds": REFERENCE_START_SECONDS,
        "max_seconds": REFERENCE_MAX_SECONDS,
        "sample_rate": 44100,
        "saved_reference_copy_to_drive": copied_to_drive,
    }
    write_json(MANIFESTS_DIR / f"reference_{voice_id}.json", reference_manifest)
    print(reference_manifest)
else:
    print("Set REFERENCE_AUDIO_PATH and REFERENCE_TEXT to run voice cloning.")


In [ ]:
#@title 7. Reference voice TTS smoke
REFERENCE_TTS_TEXT = "This is a short reference voice smoke test."  #@param {type:"string"}

if reference_manifest is None:
    print("Skipping reference TTS because no reference voice was prepared.")
else:
    ref_output = OUTPUTS_DIR / f"reference_{voice_id}_smoke.wav"
    payload = {
        "text": REFERENCE_TTS_TEXT,
        "references": [],
        "reference_id": voice_id,
        "format": "wav",
        "latency": "normal",
        "max_new_tokens": 1024,
        "chunk_length": 300,
        "top_p": 0.8,
        "repetition_penalty": 1.1,
        "temperature": 0.8,
        "streaming": False,
        "use_memory_cache": "on",
        "seed": 42,
    }
    started = time.time()
    response = requests.post(f"http://127.0.0.1:{API_PORT}/v1/tts", json=payload, timeout=600)
    elapsed = time.time() - started
    if response.status_code != 200:
        error_path = ref_output.with_suffix(".error.json")
        write_json(error_path, {"status_code": response.status_code, "body": response.text[:4000], "payload": payload, "elapsed_seconds": elapsed})
        raise RuntimeError(f"Reference TTS failed with HTTP {response.status_code}; see {error_path}")
    ref_output.write_bytes(response.content)
    manifest = {"output_path": str(ref_output), "bytes": len(response.content), "elapsed_seconds": elapsed, "reference_id": voice_id, "seed": 42, "text_chars": len(REFERENCE_TTS_TEXT)}
    write_json(ref_output.with_suffix(".wav.manifest.json"), manifest)
    print(manifest)
    display(Audio(filename=str(ref_output)))


In [ ]:
#@title 8. Launch upstream web demo after API smoke
WEB_DEMO_MODE = "gradio"  #@param ["none", "gradio", "awesome"]
STOP_API_BEFORE_WEBUI = True  #@param {type:"boolean"}
WEB_PORT = 7860
active_web_url = None
web_proc = None

if WEB_DEMO_MODE != "none" and STOP_API_BEFORE_WEBUI and "api_proc" in globals():
    stop_process(api_proc)
    print("Stopped API smoke process before launching web demo.")

if WEB_DEMO_MODE == "gradio":
    WEB_PORT = 7860
    web_proc = start_gradio_webui(compile_model=COMPILE_MODEL, half=USE_HALF)
    active_web_url = f"http://127.0.0.1:{WEB_PORT}"
    wait_for_http(active_web_url, timeout_seconds=900)
elif WEB_DEMO_MODE == "awesome":
    WEB_PORT = 8888
    if shutil.which("npm") is None:
        raise RuntimeError("npm is required to build Awesome WebUI. Install node/npm or use WEB_DEMO_MODE='gradio'.")
    run(["npm", "install"], cwd=FISH_REPO / "awesome_webui")
    run(["npm", "run", "build"], cwd=FISH_REPO / "awesome_webui")
    if not (FISH_REPO / "awesome_webui" / "dist" / "index.html").exists():
        raise RuntimeError("Awesome WebUI build did not create awesome_webui/dist/index.html")
    web_proc = start_api_server(port=WEB_PORT, compile_model=COMPILE_MODEL, half=USE_HALF)
    wait_for_http(f"http://127.0.0.1:{WEB_PORT}/v1/health", timeout_seconds=900)
    active_web_url = f"http://127.0.0.1:{WEB_PORT}/ui"
    wait_for_http(active_web_url, timeout_seconds=120)

if active_web_url:
    print("Web demo ready:", active_web_url)
    print("Web log:", web_proc["log_path"])
else:
    print("Web demo not started.")


In [ ]:
#@title 9. Optional Cloudflare named tunnel
START_CLOUDFLARE_TUNNEL = False  #@param {type:"boolean"}
CLOUDFLARED_CUSTOM_DOMAIN = ""  #@param {type:"string"}

tunnel_proc = None
if START_CLOUDFLARE_TUNNEL:
    if not active_web_url:
        raise RuntimeError("Start a web demo before launching the tunnel.")
    if shutil.which("cloudflared") is None:
        Path("/content/bin").mkdir(parents=True, exist_ok=True)
        run([
            "curl", "-L", "--fail", "--output", "/content/bin/cloudflared",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ])
        run(["chmod", "+x", "/content/bin/cloudflared"])
        os.environ["PATH"] = "/content/bin:" + os.environ["PATH"]
    tunnel_token = read_colab_secret("CLOUDFLARED_TUNNEL_TOKEN", required=True)
    tunnel_proc = start_process(
        "cloudflared",
        ["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", tunnel_token],
        cwd=WORK_ROOT,
        log_path=LOGS_DIR / "cloudflared.log",
        display_command=["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", "REDACTED"],
    )
    print("Cloudflare named tunnel started. PID:", tunnel_proc["pid"])
    print("Tunnel log:", tunnel_proc["log_path"])
    if CLOUDFLARED_CUSTOM_DOMAIN.strip():
        print("Expected public URL:", CLOUDFLARED_CUSTOM_DOMAIN.strip())
else:
    print("Tunnel skipped.")


In [ ]:
#@title 10. Inspect logs and final artifact manifest
manifest = {
    "run_id": RUN_ID,
    "run_dir": str(RUN_DIR),
    "outputs_dir": str(OUTPUTS_DIR),
    "logs_dir": str(LOGS_DIR),
    "fish_repo": str(FISH_REPO),
    "fish_commit": fish_commit,
    "active_web_url": active_web_url,
    "web_demo_mode": WEB_DEMO_MODE,
}
write_json(MANIFESTS_DIR / "final_manifest.json", manifest)
print(json.dumps(manifest, indent=2))

for log_path in sorted(LOGS_DIR.glob("*.log")):
    print("\n====", log_path, "====")
    print(read_tail(log_path, lines=40))
